# Visual Emotion Recognition — Colab runner
Runtime → Change runtime type → **T4 GPU**.

By default outputs stay inside this Colab session (`/content/emotion_out`) and **Google Drive is not used**. Download the zip at the end before closing the tab, because Colab deletes its files when the session ends.

In [ ]:
!nvidia-smi -L

USE_DRIVE = False   # True = keep outputs in Google Drive (survives disconnects)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
REPO = '/content/computer-vision'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/Renad87/computer-vision.git {REPO}
%cd {REPO}
!git checkout -q feature/emotion-training && git pull -q
!pip install -q -r ml/emotion/requirements.txt
!ls ml/emotion

In [ ]:
import os
REPO = '/content/computer-vision'
os.chdir(REPO)   # always run from the repo, even after a restart
OUT = '/content/drive/MyDrive/poetry_vision_emotion' if globals().get('USE_DRIVE') else '/content/emotion_out'
os.makedirs(OUT, exist_ok=True)
DIRS = f'--features_dir {OUT}/features --artifacts_dir {OUT}/artifacts --results_dir {OUT}/results'
CACHE = f'--cache_dir {OUT}/cache'
print('Repo ->', os.getcwd(), '| Outputs ->', OUT)

def backup(name, *folders):
    """Zip the given output folders and download them to your computer."""
    import subprocess
    from google.colab import files
    zp = f'/content/{name}.zip'
    existing = [f for f in folders if os.path.isdir(f'{OUT}/{f}')]
    if os.path.exists(zp): os.remove(zp)
    subprocess.run(['zip', '-qr', zp, *existing], cwd=OUT, check=True)
    print(f'{zp}: {os.path.getsize(zp)/1e6:.1f} MB -> downloading')
    files.download(zp)

### Restore a backup (only if Colab disconnected)
Upload a zip you downloaded earlier, then continue from the next step you have not finished. Fine-tuning also needs the cached images (`cache/`), which are not in the backups: without them, run `train.py` without `--finetune`.

In [ ]:
# from google.colab import files
# up = files.upload()
# for name in up:
#     !unzip -oq {name} -d {OUT}
# !ls {OUT}

## 1) Sample EmoSet + extract features
Faster: add `--per_class_train 600 --skip_resnet`.

In [ ]:
!python {REPO}/ml/emotion/prepare_features.py --features_dir {OUT}/features --results_dir {OUT}/results {CACHE}

In [ ]:
backup('emotion_backup_1_features', 'features', 'results')

## 2) Train: zero-shot → feature extraction → fine-tuning

In [ ]:
!python {REPO}/ml/emotion/train.py {DIRS} {CACHE} --finetune

In [ ]:
backup('emotion_backup_2_trained', 'artifacts', 'results', 'features')

## 3) Independent test evaluation

In [ ]:
!python {REPO}/ml/emotion/evaluate.py {DIRS} {CACHE}
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{OUT}/results/RESULTS.md').read()))
for f in ['model_comparison.png', 'confusion_clip_finetuned.png', 'ambiguous_examples.png', 'attributes_by_emotion.png']:
    p = f'{OUT}/results/{f}'
    if os.path.exists(p): display(Image(p))

In [ ]:
backup('emotion_backup_3_results', 'results', 'artifacts')

## 4) Try the integration function

In [ ]:
import sys, json
sys.path.insert(0, '/content/computer-vision')
from google.colab import files
up = files.upload()
from ml.emotion.inference import EmotionAnalyzer
an = EmotionAnalyzer(artifacts_dir=f'{OUT}/artifacts')
for name, data in up.items():
    print(json.dumps(an.analyze(data), ensure_ascii=False, indent=2))

## 5) Download results
Downloads `results/` and `artifacts/` (no images, no features).

In [ ]:
backup('emotion_results_final', 'results', 'artifacts')